[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peteraggi/ColabFavourites/blob/main/Task3_Replication_Rahmatia2026.ipynb)

In [ ]:
# ---------------------------------------------------------------
# CELL 1: Environment setup
# ---------------------------------------------------------------
!pip -q install kaggle scikit-learn statsmodels

import os, hashlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_recall_curve,
                              classification_report, confusion_matrix)
from statsmodels.stats.contingency_tables import mcnemar

import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)

In [ ]:
# ---------------------------------------------------------------
# CELL 2: Download RoCoLe Dataset
# ---------------------------------------------------------------

!pip -q install scikit-image kaggle

from google.colab import userdata
from pathlib import Path
import os

# ---------------------------------------------------------------
# 1. Load Kaggle credentials from Colab Secrets
# ---------------------------------------------------------------

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("Kaggle credentials loaded successfully.")

# ---------------------------------------------------------------
# 2. Download and extract RoCoLe
# ---------------------------------------------------------------

DATA_DIR = "/content"

!kaggle datasets download \
    -d nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset \
    -p /content \
    --unzip

# ---------------------------------------------------------------
# 3. Verify the dataset
# ---------------------------------------------------------------

expected_folders = [
    "coffee___healthy",
    "coffee___rust",
    "coffee___red_spider_mite"
]

print("\nDataset root:", DATA_DIR)
print("\nChecking dataset folders...")

total_images = 0

for folder in expected_folders:
    folder_path = Path(DATA_DIR) / folder

    if folder_path.exists():
        image_count = len([
            p for p in folder_path.rglob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

        total_images += image_count

        print(f"✓ {folder}: {image_count} images")
    else:
        print(f"✗ {folder}: NOT FOUND")

print(f"\nTotal images found: {total_images}")
print("DATA_DIR =", DATA_DIR)

In [ ]:
# ---------------------------------------------------------------
# CELL 3: Binary reformulation (Healthy vs Diseased)
# ---------------------------------------------------------------
# [REPLICATION CHOICE] The paper merges all rust severity levels (1-4) and red-spider-mite
# damage into a single "Diseased" class, matching its stated 791 Healthy / 769 Diseased split.
# We reproduce that merge here.

def index_dataset(root, healthy_keyword="healthy"):
    records = []
    for path in Path(root).rglob("*"):
        if path.suffix.lower() in (".jpg", ".jpeg", ".png"):
            label = "Healthy" if healthy_keyword in path.parent.name.lower() else "Diseased"
            records.append({"filepath": str(path), "label": label})
    return pd.DataFrame(records)

df = index_dataset(DATA_DIR)
print(df["label"].value_counts())
print("Original paper reports: Healthy=791, Diseased=769 (total 1560)")

## Grouped, Hash-Based 70/10/20 Split

The paper explicitly uses a **hash-based grouped split** rather than a random `train_test_split`,
to guarantee the same physical leaf/image never appears in two different splits (avoiding
leakage from near-duplicate augmented crops of the same source photo). We replicate that logic:
hash each filename to a stable integer, then bucket by the hash value into train/val/test in
fixed 70/10/20 proportions.

In [ ]:
# ---------------------------------------------------------------
# CELL 4: Hash-based grouped split (70/10/20)
# ---------------------------------------------------------------
def hash_bucket(filepath, n_buckets=1000):
    h = hashlib.md5(Path(filepath).name.encode()).hexdigest()
    return int(h, 16) % n_buckets

df["bucket"] = df["filepath"].apply(hash_bucket)
df["split"] = np.select(
    [df["bucket"] < 700, df["bucket"] < 800],
    ["train", "val"],
    default="test",
)
print(df.groupby(["split", "label"]).size())

## MobileNetV2 Feature Extraction

The original paper uses MobileNetV2 (ImageNet-pretrained) as a fixed feature extractor and
reports **256-dimensional** feature vectors per image. MobileNetV2's native
GlobalAveragePooling output is 1280-dimensional.

**[REPLICATION CHOICE]:** since the paper does not specify the exact dimensionality-reduction
layer, we reduce 1280 → 256 dimensions with **PCA fit on the training split only**, which is the
most common, reproducible way to obtain a fixed 256-d representation from a 1280-d pooled
MobileNetV2 output without introducing an untrained (i.e. randomly-initialised, non-meaningful)
dense layer.

In [ ]:
# ---------------------------------------------------------------
# CELL 5: MobileNetV2 feature extractor
# ---------------------------------------------------------------
IMG_SIZE = 224

base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                          weights="imagenet", pooling="avg")  # GlobalAveragePooling -> 1280-d
base_model.trainable = False  # frozen feature extractor, as in the original paper

def load_and_preprocess(filepath):
    img = load_img(filepath, target_size=(IMG_SIZE, IMG_SIZE))
    arr = img_to_array(img)
    return preprocess_input(arr)

def extract_batch(filepaths, batch_size=32):
    feats = []
    for i in range(0, len(filepaths), batch_size):
        batch = np.stack([load_and_preprocess(fp) for fp in filepaths[i:i+batch_size]])
        feats.append(base_model.predict(batch, verbose=0))
    return np.concatenate(feats, axis=0)

features_1280 = extract_batch(df["filepath"].tolist())
print("Raw MobileNetV2 pooled features:", features_1280.shape)

In [ ]:
# ---------------------------------------------------------------
# CELL 6: PCA reduction 1280 -> 256 dims (fit on train split only)
# ---------------------------------------------------------------
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])  # Diseased=0/Healthy=1 depending on alphabetical order
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))

train_mask = df["split"] == "train"
val_mask = df["split"] == "val"
test_mask = df["split"] == "test"

pca = PCA(n_components=256, random_state=42)
pca.fit(features_1280[train_mask.values])
X_all = pca.transform(features_1280)
print(f"Explained variance retained with 256 components: {pca.explained_variance_ratio_.sum():.4f}")

X_train, y_train = X_all[train_mask.values], df.loc[train_mask, "y"].values
X_val, y_val = X_all[val_mask.values], df.loc[val_mask, "y"].values
X_test, y_test = X_all[test_mask.values], df.loc[test_mask, "y"].values

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

## Training the Four Classifiers (as in the Original Paper)

| Classifier | Original paper's hyperparameters | Our implementation |
|---|---|---|
| Random Forest | 300 trees, bootstrap sample ratio 0.8, feature ratio 0.3 | `n_estimators=300, max_samples=0.8, max_features=0.3` |
| Linear SVM | SGD optimiser, hinge loss, lr=0.01, L2 λ=0.01 | `SGDClassifier(loss="hinge", learning_rate="constant", eta0=0.01, alpha=0.01)` wrapped in `CalibratedClassifierCV` for probability outputs (hinge loss has no native `predict_proba`) |
| KNN | not fully specified | `KNeighborsClassifier(n_neighbors=5)` (scikit-learn default, reasonable choice for ~1,100 training samples) |
| Gaussian Naive Bayes | default | `GaussianNB()` |

Following the paper, we standardise features before SVM/KNN (distance-based), but not before RF
or NB (RF is scale-invariant; GaussianNB assumes per-feature Gaussian likelihoods estimated from
the data's own scale).

In [ ]:
# ---------------------------------------------------------------
# CELL 7: Fit all four classifiers
# ---------------------------------------------------------------
scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

models = {}

models["Random Forest"] = RandomForestClassifier(
    n_estimators=300, max_samples=0.8, max_features=0.3, random_state=42, n_jobs=-1
).fit(X_train, y_train)

svm_base = SGDClassifier(loss="hinge", learning_rate="constant", eta0=0.01, alpha=0.01,
                          max_iter=1000, random_state=42)
models["Linear SVM"] = CalibratedClassifierCV(svm_base, cv=5).fit(X_train_s, y_train)

models["KNN"] = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)

models["Gaussian NB"] = GaussianNB().fit(X_train, y_train)

print("All four classifiers trained.")

## Threshold Tuning by F1-Maximisation

The original paper tunes the classification threshold (rather than using the default 0.5) to
maximise F1 on a validation set. We replicate that exactly, using our held-out validation split
from the hash-based partition above.

In [ ]:
# ---------------------------------------------------------------
# CELL 8: F1-maximising threshold search on the validation split
# ---------------------------------------------------------------
def best_f1_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])  # last point has no corresponding threshold
    return thresholds[best_idx], f1s[best_idx]

def get_features(name, split_X, split_X_s):
    return split_X_s if name in ("Linear SVM", "KNN") else split_X

results = []
preds_for_mcnemar = {}

for name, model in models.items():
    X_val_feat = get_features(name, X_val, X_val_s)
    X_test_feat = get_features(name, X_test, X_test_s)

    val_proba = model.predict_proba(X_val_feat)[:, 1]
    thresh, val_f1 = best_f1_threshold(y_val, val_proba)

    test_proba = model.predict_proba(X_test_feat)[:, 1]
    test_pred = (test_proba >= thresh).astype(int)
    preds_for_mcnemar[name] = test_pred

    results.append({
        "Model": name,
        "Threshold": round(thresh, 3),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Test F1": f1_score(y_test, test_pred),
        "Test ROC-AUC": roc_auc_score(y_test, test_proba),
    })

results_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)
results_df

## Comparison to the Original Paper's Reported Results

In [ ]:
# ---------------------------------------------------------------
# CELL 9: Side-by-side comparison table
# ---------------------------------------------------------------
original = pd.DataFrame({
    "Model": ["Gaussian NB", "Random Forest", "Linear SVM"],
    "Paper Accuracy": [0.9389, 0.9357, 0.9260],
    "Paper ROC-AUC": [np.nan, 0.9694, 0.9561],
})

comparison = results_df.merge(original, on="Model", how="left")
comparison["Accuracy Gap"] = comparison["Test Accuracy"] - comparison["Paper Accuracy"]
comparison

In [ ]:
# ---------------------------------------------------------------
# CELL 10: McNemar's test between classifier pairs (as done in the original paper)
# ---------------------------------------------------------------
from itertools import combinations

model_names = list(models.keys())
for a, b in combinations(model_names, 2):
    pred_a, pred_b = preds_for_mcnemar[a], preds_for_mcnemar[b]
    both_correct = np.sum((pred_a == y_test) & (pred_b == y_test))
    a_only = np.sum((pred_a == y_test) & (pred_b != y_test))
    b_only = np.sum((pred_a != y_test) & (pred_b == y_test))
    both_wrong = np.sum((pred_a != y_test) & (pred_b != y_test))
    table = [[both_correct, a_only], [b_only, both_wrong]]
    result = mcnemar(table, exact=True)
    print(f"{a} vs {b}: statistic={result.statistic:.3f}, p-value={result.pvalue:.4f}"
          f" -> {'significant difference' if result.pvalue < 0.05 else 'no significant difference'}")

In [ ]:
# ---------------------------------------------------------------
# CELL 11: Confusion matrices for all four models
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, preds_for_mcnemar[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_,
                yticklabels=le.classes_, ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Discussion — Does Our Replication Align With the Original Findings?

*(Fill in the actual numbers your run produced in the table above before writing this section in
the coursework document — the structure below is the analysis scaffold.)*

**Expected pattern, based on the paper's own McNemar results:** since the original study found no
statistically significant difference between RF, SVM, KNN, and NB once MobileNetV2 features are
used, a successful replication should likewise show all four classifiers clustering within a few
percentage points of each other and of the paper's 92.6-93.9% range, **not** a single model
dramatically outperforming the rest.

**Likely sources of deviation from the paper's exact numbers (document your actual observed gaps
here):**
1. **PCA vs. the paper's unspecified 1280→256 reduction** — our PCA-based reduction is a
   reasonable, reproducible stand-in, but is not guaranteed to match whatever dense/pooling layer
   the original authors used, and can shift accuracy by a few points.
2. **Hash function and bucket boundaries** — our grouped 70/10/20 split uses MD5 filename
   hashing; the paper does not publish its exact hash function, so the *specific* images landing
   in train/val/test will differ, even though the split *proportions* match.
3. **Kaggle mirror vs. original Mendeley release** — dataset mirrors can differ slightly in
   included images/folder structure from the canonical Mendeley release the paper cites.
4. **SGDClassifier(loss="hinge") vs. true SVM dual solver** — the paper explicitly describes an
   SGD-trained linear SVM (matching our implementation), but exact convergence behaviour depends
   on the number of epochs/learning-rate schedule, which is also under-specified.

**Conclusion for Task 3:** report your own accuracy/F1/AUC numbers from `comparison` above,
state whether the McNemar tests you obtained agree with the paper's "no significant difference"
conclusion, and discuss which of the four caveats above most plausibly explains any gap. This
directly satisfies Task 3's requirement to "report the results and discuss whether they align
with the original findings, noting any challenges encountered."

*Paste the shareable Colab link for this notebook into Task 3 of the coursework answer booklet.*

In [ ]:
# ---------------------------------------------------------------
# IMPROVEMENT 1: Standardise MobileNetV2 features BEFORE PCA
# ---------------------------------------------------------------
pre_pca_scaler = StandardScaler().fit(features_1280[train_mask.values])
features_1280_scaled = pre_pca_scaler.transform(features_1280)

pca_v2 = PCA(n_components=256, random_state=42)
pca_v2.fit(features_1280_scaled[train_mask.values])
X_all_v2 = pca_v2.transform(features_1280_scaled)
print(f"[Standardised PCA-256] Explained variance retained: {pca_v2.explained_variance_ratio_.sum():.4f}")
print(f"[Part 1, for comparison] Unstandardised PCA-256 explained variance was: {pca.explained_variance_ratio_.sum():.4f}")

X_train_v2, y_train_v2 = X_all_v2[train_mask.values], df.loc[train_mask, "y"].values
X_val_v2, y_val_v2 = X_all_v2[val_mask.values], df.loc[val_mask, "y"].values
X_test_v2, y_test_v2 = X_all_v2[test_mask.values], df.loc[test_mask, "y"].values

In [ ]:
# ---------------------------------------------------------------
# ABLATION A: skip PCA entirely - use the full standardised 1,280-d feature vector
# ---------------------------------------------------------------
X_train_full = features_1280_scaled[train_mask.values]
X_val_full = features_1280_scaled[val_mask.values]
X_test_full = features_1280_scaled[test_mask.values]
y_train_full, y_val_full, y_test_full = y_train_v2, y_val_v2, y_test_v2  # same labels, same split

print("Full (no-PCA) feature shapes:", X_train_full.shape, X_val_full.shape, X_test_full.shape)

In [ ]:
# ---------------------------------------------------------------
# ABLATION B: average+max pooling concatenation (2,560-d), standardised, then PCA-256
# ---------------------------------------------------------------
base_model_max = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                              weights="imagenet", pooling="max")
base_model_max.trainable = False

def extract_batch_max(filepaths, batch_size=32):
    feats = []
    for i in range(0, len(filepaths), batch_size):
        batch = np.stack([load_and_preprocess(fp) for fp in filepaths[i:i+batch_size]])
        feats.append(base_model_max.predict(batch, verbose=0))
    return np.concatenate(feats, axis=0)

features_1280_max = extract_batch_max(df["filepath"].tolist())
features_2560 = np.concatenate([features_1280, features_1280_max], axis=1)
print("Combined avg+max pooled features:", features_2560.shape)

pre_pca_scaler_gm = StandardScaler().fit(features_2560[train_mask.values])
features_2560_scaled = pre_pca_scaler_gm.transform(features_2560)

pca_gm = PCA(n_components=256, random_state=42)
pca_gm.fit(features_2560_scaled[train_mask.values])
X_all_gm = pca_gm.transform(features_2560_scaled)
print(f"[Avg+Max, standardised PCA-256] Explained variance retained: {pca_gm.explained_variance_ratio_.sum():.4f}")

X_train_gm = X_all_gm[train_mask.values]
X_val_gm = X_all_gm[val_mask.values]
X_test_gm = X_all_gm[test_mask.values]

In [ ]:
# ---------------------------------------------------------------
# IMPROVEMENT 2 & 3: proper linear SVM (SVC) + cross-validated KNN
# ---------------------------------------------------------------
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

knn_grid = GridSearchCV(
    KNeighborsClassifier(), {"n_neighbors": [3, 5, 7, 9, 11, 15]},
    cv=StratifiedKFold(5), scoring="accuracy", n_jobs=-1,
)
knn_grid.fit(X_train_v2, y_train_v2)
best_k = knn_grid.best_params_["n_neighbors"]
print("Best KNN n_neighbors (tuned on standardised-PCA features):", best_k)

def fit_and_evaluate(model, X_tr, y_tr, X_v, y_v, X_te, y_te, family, pipeline):
    model.fit(X_tr, y_tr)
    val_proba = model.predict_proba(X_v)[:, 1]
    thresh, _ = best_f1_threshold(y_v, val_proba)
    test_proba = model.predict_proba(X_te)[:, 1]
    test_pred = (test_proba >= thresh).astype(int)
    return {
        "Pipeline": pipeline,
        "Model": family,
        "Threshold": round(thresh, 3),
        "Test Accuracy": accuracy_score(y_te, test_pred),
        "Test F1": f1_score(y_te, test_pred),
        "Test ROC-AUC": roc_auc_score(y_te, test_proba),
    }, test_pred

print("Helper functions ready.")

In [ ]:
# ---------------------------------------------------------------
# Run all four classifier families across all three feature pipelines
# ---------------------------------------------------------------
pipelines = {
    "2a: Standardised PCA-256": (X_train_v2, y_train_v2, X_val_v2, y_val_v2, X_test_v2, y_test_v2),
    "2b: Full 1280-d (no PCA)": (X_train_full, y_train_full, X_val_full, y_val_full, X_test_full, y_test_full),
    "2c: Avg+Max pooling + PCA-256": (X_train_gm, y_train_v2, X_val_gm, y_val_v2, X_test_gm, y_test_v2),
}

improved_rows = []
improved_preds = {}

for pipe_name, (Xtr, ytr, Xv, yv, Xte, yte) in pipelines.items():
    rf = RandomForestClassifier(n_estimators=300, max_samples=0.8, max_features=0.3,
                                 random_state=42, n_jobs=-1)
    r, pred = fit_and_evaluate(rf, Xtr, ytr, Xv, yv, Xte, yte, "Random Forest", pipe_name)
    improved_rows.append(r); improved_preds[f"{pipe_name} | Random Forest"] = pred

    svm = SVC(kernel="linear", probability=True, class_weight="balanced", random_state=42)
    r, pred = fit_and_evaluate(svm, Xtr, ytr, Xv, yv, Xte, yte, "Linear SVM", pipe_name)
    improved_rows.append(r); improved_preds[f"{pipe_name} | Linear SVM"] = pred

    knn = KNeighborsClassifier(n_neighbors=best_k)
    r, pred = fit_and_evaluate(knn, Xtr, ytr, Xv, yv, Xte, yte, "KNN", pipe_name)
    improved_rows.append(r); improved_preds[f"{pipe_name} | KNN"] = pred

    nb_model = GaussianNB()
    r, pred = fit_and_evaluate(nb_model, Xtr, ytr, Xv, yv, Xte, yte, "Gaussian NB", pipe_name)
    improved_rows.append(r); improved_preds[f"{pipe_name} | Gaussian NB"] = pred

improved_df = pd.DataFrame(improved_rows)
improved_df.sort_values("Test Accuracy", ascending=False)

In [ ]:
# ---------------------------------------------------------------
# Grand comparison: Part 1 (faithful) vs. every Part 2 pipeline vs. the paper
# ---------------------------------------------------------------
part1_df = results_df.rename(columns={"Model": "Model"}).copy()
part1_df["Pipeline"] = "1: Faithful replication (unstandardised PCA)"

grand = pd.concat([part1_df, improved_df], ignore_index=True)
grand = grand.merge(original.rename(columns={"Model": "Model"}), on="Model", how="left")
grand["Accuracy Gap (pp)"] = (grand["Test Accuracy"] - grand["Paper Accuracy"]) * 100

grand_sorted = grand.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
grand_sorted[["Pipeline", "Model", "Test Accuracy", "Paper Accuracy", "Accuracy Gap (pp)",
              "Test ROC-AUC", "Paper ROC-AUC"]]

In [ ]:
# ---------------------------------------------------------------
# Which single pipeline got closest to the paper, on average across all 4 classifiers?
# ---------------------------------------------------------------
by_pipeline = grand.groupby("Pipeline")["Test Accuracy"].mean().sort_values(ascending=False)
print("Mean test accuracy by pipeline (across all 4 classifiers):")
print(by_pipeline)
print(f"\nPaper's mean reported accuracy (RF/SVM/NB, CNN excluded): {original['Paper Accuracy'].mean():.4f}")